In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
(
  SELECT *
  FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
  )
  WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
);
select count(distinct patient_id) from MPSII_TREATMENT_TABLE

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified AS
(
  SELECT *
  FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service,
      PROCEDURE_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service,
      NULL AS PROCEDURE_CODE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
);
select count(distinct patient_id) from MPSII_1Dx_Specified

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Specified AS
SELECT
  PATIENT_ID
FROM MPSII_1Dx_Specified
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;
select count(distinct patient_id) from MPSII_2Dx_Specified

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Tx_Specified_Tx_claims AS
SELECT *
FROM MPSII_TREATMENT_TABLE
WHERE PATIENT_ID IN (
  SELECT PATIENT_ID
  FROM MPSII_2Dx_Specified
);
select COUNT(DISTINCT patient_id) from MPSII_2Dx_Tx_Specified_Tx_claims

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Unspecified AS
(
  SELECT *
  FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
);
select count(distinct patient_id) from MPSII_1Dx_Unspecified

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Unspecified
AS 
(SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM MPSII_1Dx_Unspecified--TABLE
--WHERE ARRAYS_OVERLAP (SPLIT(DIAGNOSIS_CODES, '|'), ARRAY_CONSTRUCT_COMPACT('E761'))
--WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31' 
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2);
select count(distinct patient_id) from MPSII_2Dx_Unspecified

In [0]:
%sql
-- MPSII Unspecified 2 Dx Elaprase Treated Patients
Select count(distinct patient_id) from MPSII_2Dx_Unspecified
where patient_id in (
Select distinct patient_id from MPSII_TREATMENT_TABLE
where code in ('54092070001','540920700','J1743')
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_Incremental_Patients AS
SELECT *
FROM MPSII_2Dx_Unspecified
WHERE patient_id IN (
  SELECT DISTINCT patient_id FROM MPSII_TREATMENT_TABLE
  WHERE code IN ('54092070001','540920700','J1743')
)
AND patient_id NOT IN (
  SELECT DISTINCT patient_id FROM MPSII_2Dx_Tx_Specified_Tx_claims
);

-- Just read the count:
select count(distinct patient_id) from mpsii_incremental_patients


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_all_Dx_1167 AS
SELECT a.*,
       b.HCO_PRIMARY_NPI,
       b.PRIMARY_SPECIALTY,
       b.SECONDARY_SPECIALTY,
       CASE 
         WHEN primary_specialty LIKE '%Genetic%' OR secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
         WHEN primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
         WHEN primary_specialty LIKE '%Psychiatry & Neurology%' OR secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
              primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
         WHEN primary_specialty LIKE '%Nurse Practitioner%' OR primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
         WHEN primary_specialty LIKE '%Internal Medicine%' OR secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
         WHEN primary_specialty LIKE '%Family Medicine%' OR secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
         WHEN a.npi IS NULL THEN 'NA'
         ELSE 'Others'
       END AS SPECIALTY
FROM (
  SELECT * FROM MPSII_1Dx_Specified
  WHERE patient_id IN (SELECT DISTINCT patient_id FROM MPSII_2Dx_Specified)

  UNION

  SELECT *, null as null FROM MPSII_1Dx_Unspecified
  WHERE patient_id IN (SELECT DISTINCT patient_id FROM MPSII_Incremental_Patients)
) a
LEFT JOIN com_edp_prd.com_raw.kom_providers b
  ON a.npi = b.npi
;
select count(distinct patient_id) from mpsii_all_dx_1167

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW Dx_1_1_Mapped_HCP_Refresh AS
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    'DX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date, 
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM MPSII_all_Dx_1167
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
);

In [0]:
%sql
SELECT NPI, `Age bucket`, COUNT(DISTINCT N_PATS) AS patient_count
FROM (
    SELECT *,
           CASE 
               WHEN age BETWEEN 0 AND 10 THEN '0-10'
               WHEN age > 10 THEN '11+'
		ELSE 'NA'
           END AS `Age bucket`
    FROM (
        SELECT A.*, 
               B.PATIENT_YOB, 
               YEAR(A.MIN_FILL_DATE) - YEAR(B.PATIENT_YOB) AS age
        FROM (
            SELECT NPI, N_PATS, MIN(FILL_DATE) AS MIN_FILL_DATE
            FROM Dx_1_1_Mapped_HCP_Refresh
            GROUP BY NPI, N_PATS
        ) AS A
        LEFT JOIN com_raw.kom_patient_demographics B
        ON A.N_PATS = B.PATIENT_ID
    )
)
GROUP BY NPI, `Age bucket`;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_Before_2024 AS
(
  SELECT *
  FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL AS Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE < '2024-01-01'
);
select count(distinct patient_id) from MPSII_1Dx_Specified_Before_2024

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE_Before_2024 AS 
(
  SELECT * 
  FROM (
    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,   
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE, 
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
  )
  WHERE FILL_DATE < '2024-01-01'
);
select count(distinct patient_id) from MPSII_TREATMENT_TABLE_Before_2024

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified_Before_2025 AS 
(
  SELECT *
  FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE < '2025-01-01'
);
select count(distinct patient_id) from MPSII_1Dx_Specified_Before_2025

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE_Before_2025 AS 
(
  SELECT * 
  FROM (
    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,   
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE, 
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
  )
  WHERE FILL_DATE < '2025-01-01'
);
select count(distinct patient_id) from MPSII_TREATMENT_TABLE_Before_2025

In [0]:
%sql
SELECT NPI, `Age bucket`, COUNT(DISTINCT N_PATS) AS patient_count
FROM (
    SELECT *,
           CASE 
               WHEN age BETWEEN 0 AND 10 THEN '0-10'
               WHEN age > 10 THEN '11+'
               Else 'NA'
           END AS `Age bucket`
    FROM (
        SELECT A.*, 
               B.PATIENT_YOB, 
               YEAR(A.MIN_FILL_DATE) - YEAR(B.PATIENT_YOB) AS age
        FROM (
            SELECT NPI, N_PATS, MIN(FILL_DATE) AS MIN_FILL_DATE
            FROM Dx_1_1_Mapped_HCP_Refresh
            GROUP BY NPI, N_PATS
        ) AS A
        LEFT JOIN com_raw.kom_patient_demographics B
        ON A.N_PATS = B.PATIENT_ID
    )
)
where N_PATS not in (Select distinct patient_id from MPSII_1Dx_Specified_Before_2024)
and N_PATS in (Select distinct patient_id from MPSII_1Dx_Specified_Before_2025)
GROUP BY NPI, `Age bucket`;

In [0]:
%sql
SELECT HCO_PRIMARY_NPI, `Age bucket`, COUNT(DISTINCT N_PATS) AS patient_count
FROM (
    SELECT *,
           CASE 
               WHEN age BETWEEN 0 AND 10 THEN '0-10'
               WHEN age > 10 THEN '11+'
               else 'NA'
           END AS `Age bucket`
    FROM (
        SELECT A.*, 
               B.PATIENT_YOB, 
               YEAR(A.MIN_FILL_DATE) - YEAR(B.PATIENT_YOB) AS age
        FROM (
            SELECT HCO_PRIMARY_NPI, N_PATS, MIN(FILL_DATE) AS MIN_FILL_DATE
            FROM Dx_1_1_Mapped_HCP_Refresh A
            LEFT JOIN com_raw.kom_providers b
            on a.npi = b.npi
            GROUP BY HCO_PRIMARY_NPI, N_PATS
        ) AS A
        LEFT JOIN com_raw.kom_patient_demographics B
        ON A.N_PATS = B.PATIENT_ID
    )
)
where N_PATS not in (Select distinct patient_id from MPSII_1Dx_Specified_Before_2024)
and N_PATS in (Select distinct patient_id from MPSII_1Dx_Specified_Before_2025)
GROUP BY HCO_PRIMARY_NPI, `Age bucket`;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE_2023_to_25 AS 
(
  SELECT * 
  FROM (
    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID as EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT 
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,   
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE, 
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
  )
  WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
);
select count(distinct patient_id) from MPSII_TREATMENT_TABLE_2023_to_25

In [0]:
%sql
select distinct patient_id from mpsii_treatment_table_2023_to_25
where patient_id in (select patient_id from mpsII_2Dx_specified)

In [0]:
%sql
select count(distinct patient_id) from mpsII_treatment_table
where patient_id not in (select patient_id from mpsII_2Dx_specified)

In [0]:
%sql
SELECT DISTINCT s.procedure_code
FROM MPSII_1Dx_Specified s
WHERE s.PATIENT_ID IN (SELECT PATIENT_ID FROM MPSII_2Dx_Specified)
  AND s.procedure_code ILIKE '%J%';
